# Submissão 2 - Modelo A (Numpy DNN / Modelo Antigo)
Notebook preparado para ler o dataset de validação e exportar a classificação usando o Modelo A (da Submissão 1).

In [ ]:
import pickle
import pandas as pd
import numpy as np
import re
from collections import Counter
import sys
import os

sys.path.append("../src")
sys.path.append("../models")

with open("../subm1/modelo_numpy_submA.pkl", "rb") as f:
    artefactos = pickle.load(f)

model_binary = artefactos["modelo_binario"]
model_llm = artefactos["modelo_llm"]
vocab = artefactos["vocabulario"]
idf = artefactos["idf"]
llm_mapping = artefactos["llm_mapping"]

print("Modelo carregado com sucesso.")

In [ ]:
STOPWORDS = {
    "the", "a", "an", "and", "or", "of", "to", "in", "on", "at", "for", "with",
    "is", "are", "was", "were", "be", "been", "being", "this", "that", "these",
    "those", "it", "its", "as", "by", "from", "but", "about", "into", "than",
    "then", "so", "such", "if", "their", "there", "they", "them", "he", "she",
    "you", "your", "we", "our", "i", "my", "me", "his", "her", "what", "which",
    "who", "whom", "can", "could", "should", "would", "do", "does", "did", "have",
    "has", "had", "not", "no", "yes", "will", "just"
}

def tokenize(text):
    text = text.lower()
    tokens = re.findall(r"\b[a-zA-ZÀ-ÿ]{2,}\b", text)
    return [tok for tok in tokens if tok not in STOPWORDS]

def vectorize_tfidf(texts, vocab, idf):
    X = np.zeros((len(texts), len(vocab)), dtype=np.float64)
    for i, text in enumerate(texts):
        tokens = [tok for tok in tokenize(text) if tok in vocab]
        if not tokens:
            continue
        counts = Counter(tokens)
        total_terms = len(tokens)
        for tok, count in counts.items():
            j = vocab[tok]
            tf = count / total_terms
            X[i, j] = tf * idf[j]
    return X

def l2_normalize_rows(X, eps=1e-12):
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    return X / (norms + eps)

class SimpleDataset:
    def __init__(self, X, y):
        self.X = X
        self.y = y

In [ ]:
# Ler CSV original da subm2 (ex: dataset_teste.csv)
df_subm2 = pd.read_csv("../data/dataset_teste.csv")
X_subm2_text = df_subm2["Text"].fillna("").astype(str).values
X_subm2 = vectorize_tfidf(X_subm2_text, vocab, idf)
X_subm2 = l2_normalize_rows(X_subm2)

dummy = SimpleDataset(X_subm2, np.zeros(len(X_subm2), dtype=int))

In [ ]:
prob_bin = model_binary.predict(dummy)
pred_bin = (prob_bin >= 0.512).astype(int).flatten()

indices_llm = np.where(pred_bin == 1)[0]
X_llm = X_subm2[indices_llm]

if len(indices_llm) > 0:
    dummy_llm = SimpleDataset(X_llm, np.zeros(len(X_llm), dtype=int))
    prob_llm = model_llm.predict(dummy_llm)
    pred_llm = np.argmax(prob_llm, axis=1)
else:
    pred_llm = np.array([], dtype=int)

In [ ]:
final_labels = np.array(["Human"] * len(df_subm2), dtype=object)

for pos, pred in zip(indices_llm, pred_llm):
    final_labels[pos] = llm_mapping[int(pred)]

df_output = df_subm2.copy()
df_output["Labels"] = final_labels

df_output.to_csv("subm2-MIA-A.csv", index=False)
print("CSV Modelo A criado com sucesso.")
df_output.head()